# 04 — Target encoding du destinataire (hypothèse collecteur)

Les features destinataire dominent l'importance (notebook 03). Hypothèse : les comptes
**destinataires** sont moins mixtes que les émetteurs (collecteurs de fraude). Si c'est vrai,
un **target encoding fold-safe du destinataire** (taux de fraude historique, appris sur le passé)
devrait apporter un vrai gain — c'est la version légitime de l'encodage d'ID interdit pour les émetteurs.

On part du jeu de features complet du notebook 03 et on mesure le gain en CV temporelle (échelle LB).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import target_encode_ref
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
tr03 = train[op03].reset_index(drop=True)

## Diagnostic : les destinataires sont-ils mixtes ? (vs émetteurs 96%)

In [ ]:
def purity(df, col):
    g = df.groupby(col)[C.TARGET].mean()
    prof = pd.cut(g, [-0.01, 0.0, 0.999, 1.01], labels=["jamais", "mixte", "toujours"])
    tx = df.groupby(col).size().groupby(prof, observed=False).sum()
    return (100 * tx / tx.sum()).round(1)

print("% des tx op_03 par profil ÉMETTEUR :")
print(purity(tr03, C.ORIGIN_ACCT))
print("\n% des tx op_03 par profil DESTINATAIRE :")
print(purity(tr03, C.DEST_ACCT))

In [ ]:
EPS = 1e-6
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def build_features(df, ref, te_cols=()):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    X = pd.concat([X, beh, rec], axis=1)
    for col in te_cols:
        X[f"te_{col}"] = target_encode_ref(df.reset_index(drop=True), ref, col, C.TARGET)
    return X

def make_model():
    try:
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                                  learning_rate=0.05, iterations=600, random_seed=42, verbose=False)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=42)

## A/B : features 03 seules vs + target encoding destinataire
Le TE est appris **fold par fold sur le passé** (ref = train du fold) -> anti-fuite.

In [ ]:
folds_full = list(time_folds(train[C.PERIOD]))

def run_cv(te_cols):
    oof = np.zeros(len(train))
    per_fold = []
    last_m, last_c = None, None
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]
        va_op = va_idx[op03[va_idx]]
        ref = train.iloc[tr_op]
        Xtr = build_features(train.iloc[tr_op], ref, te_cols)
        Xva = build_features(train.iloc[va_op], ref, te_cols)
        m = make_model(); m.fit(Xtr, y_all[tr_op])
        oof[va_op] = m.predict_proba(Xva)[:, 1]
        per_fold.append(evaluate_ap(y_all[va_op], oof[va_op]))
        last_m, last_c = m, Xtr.columns
    return per_fold, last_m, last_c

pf_base, _, _ = run_cv(te_cols=())
pf_dest, m_dest, c_dest = run_cv(te_cols=[C.DEST_ACCT])
pf_both, _, _ = run_cv(te_cols=[C.DEST_ACCT, C.ORIGIN_ACCT])

def show(name, pf):
    print(f"{name:28s} global {np.mean(pf):.4f} | recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f}")
show("03 features (sans TE)", pf_base)
show("+ TE destinataire", pf_dest)
show("+ TE destinataire+émetteur", pf_both)
print("\nGain TE dest sur last fold :", round(pf_dest[-1] - pf_base[-1], 4))

In [ ]:
imp = m_dest.get_feature_importance() if hasattr(m_dest, "get_feature_importance") else m_dest.feature_importances_
print(pd.Series(imp, index=c_dest).sort_values(ascending=False).round(2).head(12))

In [ ]:
# Soumission avec TE destinataire + émetteur (le candidat gagnant)
from src.calibration import fit_isotonic, apply_isotonic
from src.utils import make_submission
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
TE = [C.DEST_ACCT, C.ORIGIN_ACCT]

# OOF (pour calibrer) avec les TE
oof = np.zeros(len(train))
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    m = make_model(); m.fit(build_features(train.iloc[tr_op], ref, TE), y_all[tr_op])
    oof[va_op] = m.predict_proba(build_features(train.iloc[va_op], ref, TE))[:, 1]

# modèle final sur tout le train op_03
ref_full = train.iloc[np.where(op03)[0]]
final = make_model(); final.fit(build_features(ref_full, ref_full, TE), y_all[op03])
iso = fit_isotonic(oof[op03], y_all[op03])

# scoring test : TE appris sur tout le train (pas de fuite, labels test inconnus)
te_op = op03_mask(test).to_numpy()
proba = apply_isotonic(iso, final.predict_proba(build_features(test.iloc[np.where(te_op)[0]], ref_full, TE))[:, 1])
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "04_te_dest_origin")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))

## Décision
- Si **TE destinataire** fait gagner nettement (last fold > +0.01) ET que TE émetteur dégrade →
  hypothèse collecteur confirmée : on garde le TE destinataire, on régénère la soumission, on soumet.
- Si gain marginal → les destinataires sont aussi mixtes ; on cherche ailleurs (interactions,
  autres agrégations destinataire : montant médian reçu, écart-type, ratio entrant/sortant).